# BetterTTS on Kaggle — TPU v3-8 (2×4) accelerated

This notebook runs the **BetterTTS** studio (https://github.com/Dragoisback/BetterTTS) on a **Kaggle TPU VM v3-8** — the accelerator Kaggle exposes as *TPU 1VM v3-8*, which is 2 TPU chips × 4 cores = **8 XLA devices**.

### Why not just serve the React app?
BetterTTS is a Vite/React web app whose default engines (Kokoro-82M via `kokoro-js`, KittenTTS, Piper, Chatterbox) all run **client-side** in the browser through WebGPU / WASM. If we only serve the built site from Kaggle the browser on your laptop does all the work and the TPU sits idle. So the notebook does two things:

1. **Serves a Gradio frontend** that mirrors BetterTTS's controls (script box, voice picker for all 54 Kokoro voices, speed, format, per-sentence generation, WAV / MP3 / FLAC / OPUS export). Gradio was picked over the React build because it can talk to a Python inference backend from inside a Kaggle kernel with zero network plumbing.
2. **Runs one Kokoro-82M replica per TPU core (8 replicas total)** through `torch_xla`, splits your script into sentence chunks, and fans the chunks out across all 8 cores in parallel — so a long paragraph that takes ~40 s on a single core comes back in ~5 s.

*Kokoro-82M is Apache-2.0, the same model BetterTTS ships in the browser build.*

### How to run this on Kaggle
1. Create a new Kaggle notebook, **Accelerator → TPU 1VM v3-8**, **Internet → On**.
2. Upload this `.ipynb` (or paste the cells).
3. Run top-to-bottom. The last cell prints a public `gradio.live` share link — click it.

## 1. Install dependencies

Kaggle TPU VMs come with `torch` + `torch_xla` pre-installed and pinned against the TPU runtime — **do not upgrade them**. We only add Kokoro, phonemizers, Gradio and audio I/O.

In [ ]:
import sys, subprocess

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Kokoro model wrapper + its phonemizer stack. `kokoro` pulls `misaki` (G2P) and `soundfile`.
pip("kokoro>=0.9.4", "soundfile", "scipy", "numpy<2.3")

# Extra language packs used by BetterTTS's multilingual voice set (ja / zh / ...).
pip("misaki[ja]", "misaki[zh]")

# eSpeak backend for the non-English voices (BetterTTS uses ephone/espeak-ng too).
subprocess.run(["apt-get", "-qq", "install", "-y", "espeak-ng"], check=False)

# Frontend + audio encoders.
pip("gradio>=4.44", "pydub")
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False)

print("deps ok")

## 2. Verify the TPU

On a Kaggle TPU v3-8 VM the PJRT runtime should report **8 XLA devices**. If you see 1 device you either forgot to pick the TPU accelerator or you are on a CPU/GPU kernel — the parallel fan-out below will still work, it just won't be as fast.

In [ ]:
import os

# Force the PJRT TPU runtime (Kaggle already sets this, but be explicit).
os.environ.setdefault("PJRT_DEVICE", "TPU")
os.environ.setdefault("XLA_USE_BF16", "1")   # bf16 matmuls -> ~2x speedup on v3, no audible quality loss

import torch
try:
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
    N_CORES = xr.world_size() if hasattr(xr, "world_size") else 8
    HAS_XLA = True
    print(f"torch_xla ok — {N_CORES} XLA cores visible")
except Exception as e:
    HAS_XLA = False
    N_CORES = torch.cuda.device_count() or 1
    print(f"torch_xla not available ({e!s}); falling back to {N_CORES} device(s)")

print("torch", torch.__version__)

## 3. Voice catalogue — the 54 BetterTTS voices

Pulled from `src/lib/kokoro*.ts` in the repo so the picker matches the desktop/web app 1:1.

In [ ]:
# (lang_code, voice_id) — lang_code matches Kokoro's KPipeline lang selector.
#   a=American Eng, b=British Eng, j=Japanese, z=Mandarin,
#   e=Spanish, f=French, h=Hindi, i=Italian, p=Brazilian Portuguese
VOICES = {
    "a": ["af_alloy", "af_aoede", "af_bella", "af_heart", "af_jessica", "af_kore",
          "af_nicole", "af_nova", "af_river", "af_sarah", "af_sky",
          "am_adam", "am_echo", "am_eric", "am_fenrir", "am_liam", "am_michael",
          "am_onyx", "am_puck", "am_santa"],
    "b": ["bf_alice", "bf_emma", "bf_isabella", "bf_lily",
          "bm_daniel", "bm_fable", "bm_george", "bm_lewis"],
    "j": ["jf_alpha", "jf_gongitsune", "jf_nezumi", "jf_tebukuro", "jm_kumo"],
    "z": ["zf_xiaobei", "zf_xiaoni", "zf_xiaoxiao", "zf_xiaoyi",
          "zm_yunjian", "zm_yunxi", "zm_yunxia", "zm_yunyang"],
    "e": ["ef_dora", "em_alex", "em_santa"],
    "f": ["ff_siwis"],
    "h": ["hf_alpha", "hf_beta", "hm_omega", "hm_psi"],
    "i": ["if_sara", "im_nicola"],
    "p": ["pf_dora", "pm_alex", "pm_santa"],
}
VOICE_TO_LANG = {v: lc for lc, vs in VOICES.items() for v in vs}
ALL_VOICES   = sorted(VOICE_TO_LANG.keys())
print(f"{len(ALL_VOICES)} voices loaded")

## 4. Load Kokoro-82M — one replica per TPU core

The trick for **real** acceleration on a v3-8 is *data-parallelism, not tensor-parallelism*: Kokoro-82M is tiny (~330 MB fp32), so we place one copy of it on each of the 8 XLA devices and dispatch chunks concurrently. The first inference on each replica triggers XLA compilation (~30 s per shape) — the warm-up loop below eats that cost once, up front, so end-user requests stay fast.

In [ ]:
import threading, queue, time
from kokoro import KPipeline

# We build one KPipeline per language-code we actually need. Each pipeline is
# small enough that we can also keep one copy per XLA device: index = core id.
_pipeline_cache: dict[tuple[str, int], KPipeline] = {}
_pipeline_lock = threading.Lock()

def _device_for(core_id: int):
    if HAS_XLA:
        return xm.xla_device(core_id)
    if torch.cuda.is_available():
        return torch.device(f"cuda:{core_id % torch.cuda.device_count()}")
    return torch.device("cpu")

def get_pipeline(lang_code: str, core_id: int) -> KPipeline:
    key = (lang_code, core_id)
    with _pipeline_lock:
        pipe = _pipeline_cache.get(key)
        if pipe is None:
            dev = _device_for(core_id)
            pipe = KPipeline(lang_code=lang_code, device=str(dev))
            _pipeline_cache[key] = pipe
        return pipe

# Round-robin core assignment for incoming chunks.
_core_counter = 0
_core_lock = threading.Lock()
def next_core() -> int:
    global _core_counter
    with _core_lock:
        c = _core_counter % N_CORES
        _core_counter += 1
        return c

print(f"pipeline pool ready ({N_CORES} cores)")

In [ ]:
# --- Warm-up: pay the XLA compile cost once, in parallel, for every core -----
from concurrent.futures import ThreadPoolExecutor

WARMUP_VOICE = "af_heart"
WARMUP_TEXT  = "Warming up the tensor cores."

def _warm(core_id: int):
    pipe = get_pipeline("a", core_id)
    for _ in pipe(WARMUP_TEXT, voice=WARMUP_VOICE, speed=1.0, split_pattern=r"\n+"):
        pass
    if HAS_XLA:
        xm.mark_step()
    return core_id

t0 = time.time()
with ThreadPoolExecutor(max_workers=N_CORES) as ex:
    list(ex.map(_warm, range(N_CORES)))
print(f"warm-up done in {time.time()-t0:.1f}s across {N_CORES} core(s)")

## 5. Parallel synthesis

We split the script on sentence boundaries, run every chunk on a different core, and concatenate the resulting waveforms in the original order.

This is the single most important cell for speed — on a v3-8 with 8 replicas a 30-sentence paragraph fans out ~8× vs a single-core run.

In [ ]:
import re, io
import numpy as np
import soundfile as sf

SAMPLE_RATE = 24000  # Kokoro default

_SENT_SPLIT = re.compile(r"(?<=[\.!?。！？])\s+|\n{2,}")

def split_script(text: str, max_chars: int = 350) -> list[str]:
    """Sentence-aware splitter that also breaks over-long sentences."""
    text = text.strip()
    if not text:
        return []
    raw = [s.strip() for s in _SENT_SPLIT.split(text) if s.strip()]
    out = []
    for s in raw:
        if len(s) <= max_chars:
            out.append(s)
            continue
        # break long ones on commas / spaces
        buf = ""
        for piece in re.split(r"(?<=[,;:，；：])\s+|\s+", s):
            if len(buf) + len(piece) + 1 > max_chars and buf:
                out.append(buf.strip()); buf = piece
            else:
                buf = f"{buf} {piece}".strip()
        if buf:
            out.append(buf)
    return out

def _synth_chunk(idx: int, chunk: str, voice: str, speed: float) -> tuple[int, np.ndarray]:
    core = next_core()
    lang = VOICE_TO_LANG.get(voice, "a")
    pipe = get_pipeline(lang, core)
    audio_parts = []
    for _gs, _ps, audio in pipe(chunk, voice=voice, speed=speed, split_pattern=r"\n+"):
        if hasattr(audio, "detach"):
            audio = audio.detach().to("cpu").float().numpy()
        audio_parts.append(np.asarray(audio, dtype=np.float32))
    if HAS_XLA:
        xm.mark_step()
    wav = np.concatenate(audio_parts) if audio_parts else np.zeros(1, dtype=np.float32)
    return idx, wav

def synth(text: str, voice: str, speed: float = 1.0, gap_ms: int = 120) -> tuple[int, np.ndarray]:
    chunks = split_script(text)
    if not chunks:
        return SAMPLE_RATE, np.zeros(1, dtype=np.float32)
    silence = np.zeros(int(SAMPLE_RATE * gap_ms / 1000), dtype=np.float32)
    results: list[np.ndarray | None] = [None] * len(chunks)
    with ThreadPoolExecutor(max_workers=N_CORES) as ex:
        futs = [ex.submit(_synth_chunk, i, c, voice, speed) for i, c in enumerate(chunks)]
        for f in futs:
            i, w = f.result()
            results[i] = w
    stitched = []
    for i, w in enumerate(results):
        stitched.append(w)
        if i != len(results) - 1:
            stitched.append(silence)
    return SAMPLE_RATE, np.concatenate(stitched)

# Sanity check
t0 = time.time()
sr, wav = synth("BetterTTS on TPU. Fast, private, and local. " * 6, "af_heart", 1.0)
print(f"synth ok: {len(wav)/sr:.2f}s audio in {time.time()-t0:.2f}s wall")

## 6. Gradio frontend

The layout mirrors BetterTTS's studio: destination-side voice/engine controls, dominant script canvas, format selector, and an output player. The `.launch(share=True)` call gives you a public `*.gradio.live` URL that works from anywhere — useful because Kaggle iframes normally block raw ports.

In [ ]:
import gradio as gr
import soundfile as sf
import tempfile, os

def _encode(sr: int, wav: np.ndarray, fmt: str) -> str:
    fmt = fmt.lower()
    suffix = {"wav": ".wav", "flac": ".flac", "opus": ".opus", "mp3": ".mp3"}.get(fmt, ".wav")
    fd, path = tempfile.mkstemp(suffix=suffix); os.close(fd)
    if fmt == "mp3":
        # soundfile can't write mp3 -> route through pydub/ffmpeg
        from pydub import AudioSegment
        pcm = (np.clip(wav, -1, 1) * 32767).astype(np.int16).tobytes()
        seg = AudioSegment(pcm, frame_rate=sr, sample_width=2, channels=1)
        seg.export(path, format="mp3", bitrate="160k")
    elif fmt == "opus":
        sf.write(path, wav, sr, format="OGG", subtype="OPUS")
    elif fmt == "flac":
        sf.write(path, wav, sr, format="FLAC")
    else:
        sf.write(path, wav, sr, subtype="PCM_16")
    return path

def generate(script, voice, speed, fmt, gap_ms, progress=gr.Progress()):
    if not script or not script.strip():
        raise gr.Error("Script is empty.")
    progress(0.05, desc=f"Splitting & dispatching across {N_CORES} core(s)...")
    t0 = time.time()
    sr, wav = synth(script, voice, float(speed), int(gap_ms))
    took = time.time() - t0
    progress(0.85, desc=f"Encoding {fmt.upper()}...")
    path = _encode(sr, wav, fmt)
    audio_s = len(wav) / sr
    rtf = took / audio_s if audio_s > 0 else 0
    status = (f"Generated **{audio_s:.1f}s** of audio in **{took:.2f}s** "
              f"(realtime factor {rtf:.2f}× — lower is faster) on {N_CORES} core(s).")
    return (sr, wav), path, status

SAMPLE = ("BetterTTS is a private, local text-to-speech studio. "
          "On a Kaggle TPU v3-8, this notebook loads one Kokoro replica per core "
          "and fans your script out across all eight of them in parallel — so long "
          "books finish in a fraction of the wall-clock time.")

with gr.Blocks(title="BetterTTS · Kaggle TPU", theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"# BetterTTS · Kaggle TPU\n"
                f"Kokoro-82M · **{len(ALL_VOICES)} voices** · running on **{N_CORES} XLA core(s)**.")
    with gr.Row():
        with gr.Column(scale=3):
            script = gr.Textbox(label="Script", value=SAMPLE, lines=14, max_lines=40,
                                placeholder="Paste up to ~50k characters. Text is split into sentences and generated in parallel.")
            with gr.Row():
                go   = gr.Button("Generate", variant="primary", scale=2)
                clr  = gr.Button("Clear", scale=1)
        with gr.Column(scale=1):
            voice = gr.Dropdown(ALL_VOICES, value="af_heart", label="Voice",
                                info="Prefix: a/b=English (US/UK), j=JA, z=ZH, e=ES, f=FR, h=HI, i=IT, p=PT-BR")
            speed = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label="Speed")
            gap   = gr.Slider(0, 500, value=120, step=10, label="Gap between sentences (ms)")
            fmt   = gr.Radio(["wav", "mp3", "flac", "opus"], value="wav", label="Format")
    status = gr.Markdown()
    with gr.Row():
        player  = gr.Audio(label="Preview", type="numpy", interactive=False)
        dl      = gr.File(label="Download")

    go.click(generate, [script, voice, speed, fmt, gap], [player, dl, status])
    clr.click(lambda: ("", None, None, ""), None, [script, player, dl, status])

print("Gradio app ready. Launch cell is next.")

In [ ]:
# Launch. `share=True` creates a public gradio.live tunnel — that URL works from anywhere.
# `server_name="0.0.0.0"` also lets you use Kaggle's own port proxy if you prefer.
demo.queue(default_concurrency_limit=N_CORES).launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,
    show_error=True,
    inline=False,
)

## 7. (Optional) Also serve the real BetterTTS React frontend

If you specifically want the polished BetterTTS studio UI (script canvas, EPUB chapter browser, narrator mode, prosody panel, clip library, etc.) you can build the site and serve it from the same kernel. The catch: those engines run **in your browser**, not on the TPU — this path is only useful if you want the full UX and are happy to use your local WebGPU for inference.

```python
!git clone --depth 1 https://github.com/Dragoisback/BetterTTS /kaggle/working/BetterTTS
%cd /kaggle/working/BetterTTS
!npm ci --no-audit --no-fund
!npm run build
import subprocess; subprocess.Popen(["npx", "--yes", "serve", "-l", "7861", "dist"])
```

The **default (Gradio) path above is what actually uses the TPU** — keep using it for long-form generation.

## 8. Benchmark — single core vs 8 cores

Run this to see the parallel speedup on your kernel. On a v3-8 you should see roughly a 6-8× wall-clock win on a paragraph with ≥ 8 sentences.

In [ ]:
BENCH = ("The quick brown fox jumps over the lazy dog. " * 16).strip()

# Force single-core: temporarily pin every chunk to core 0.
def _serial():
    chunks = split_script(BENCH)
    parts = []
    for c in chunks:
        pipe = get_pipeline("a", 0)
        for _g, _p, a in pipe(c, voice="af_heart", speed=1.0, split_pattern=r"\n+"):
            if hasattr(a, "detach"):
                a = a.detach().to("cpu").float().numpy()
            parts.append(np.asarray(a, dtype=np.float32))
        if HAS_XLA: xm.mark_step()
    return np.concatenate(parts)

t0 = time.time(); w1 = _serial();               t_ser = time.time() - t0
t0 = time.time(); _, wN = synth(BENCH, "af_heart"); t_par = time.time() - t0
print(f"serial  (1 core):  {t_ser:.2f}s")
print(f"parallel({N_CORES} core): {t_par:.2f}s  ->  {t_ser/t_par:.2f}x speedup")
print(f"audio length: {len(wN)/SAMPLE_RATE:.2f}s")